# Session 4, Monty Hall: don't argue about probability, simulate it

> **Whole-class warm-up.** Three doors. Behind one is a car; behind the other two, goats. You pick a door. The host, who knows where the car is, opens a *different* door to reveal a goat, then asks: **do you want to switch to the other closed door, or stay?**

Almost everyone's gut says *"two doors left, so it's 50/50, doesn't matter."* We are not going to **argue** about it. We are going to **simulate** it, the move at the heart of this whole course: when intuition and a claim disagree, generate data and look.

Keep one question in the back of your mind: *when did the probability change, and why?* That is the same question a classifier answers, and today you'll build one.

## Predict before you compute  🎯

Commit to a view before you run anything (write it in the next cell):

- If you **always switch**, what share of games do you win?
- If you **always stay**, what share do you win?

In [ ]:
# Your prediction (just notes):
#
# Always SWITCH, I win about ___ % of games
# Always STAY, I win about ___ % of games

## Step 1, Play one game in code

> Ask Claude: *"Write a Python function that plays one game of Monty Hall and returns True if I win. The host always opens a goat door that isn't my pick. Add a `switch` argument."*

Read the function until you believe it, the whole demo rests on the host opening a **goat**, never the car and never your door.

In [ ]:
import random

def play_one(switch):
    doors = [0, 0, 0]
    doors[random.randint(0, 2)] = 1          # 1 = car, put it behind a random door
    pick = random.randint(0, 2)               # your first pick
    # host opens a goat door that is neither your pick nor the car:
    host = random.choice([d for d in range(3) if d != pick and doors[d] == 0])
    if switch:
        pick = [d for d in range(3) if d != pick and d != host][0]
    return doors[pick] == 1                    # did you land on the car?

## Step 2, Now play 10,000 games each way

One game tells you nothing (you win or you don't). Ten thousand tells you the **probability**.

In [ ]:
N = 10_000
switch_wins = sum(play_one(True)  for _ in range(N)) / N
stay_wins   = sum(play_one(False) for _ in range(N)) / N
print(f'Always SWITCH wins {switch_wins:.1%} of games')
print(f'Always STAY   wins {stay_wins:.1%} of games')

Switching wins about **2/3** of the time; staying, about **1/3**. Your gut was wrong, and it wasn't close. How did you do against your prediction?

## Step 3, Watch the probability settle

Why trust 10,000 games and not 10? Plot the running win-rate as the games pile up, it lurches around at first, then **settles** onto 2/3. (This is the same law of large numbers behind last week's Central Limit Theorem: more data → a steadier estimate.)

In [ ]:
import matplotlib.pyplot as plt
from itertools import accumulate

results = [play_one(True) for _ in range(N)]
running = [total / i for i, total in enumerate(accumulate(results), start=1)]
plt.figure(figsize=(7, 3.5))
plt.plot(running, color='navy', lw=1)
plt.axhline(2/3, color='orange', ls='--', label='2/3')
plt.xlabel('games played'); plt.ylabel('win rate (always switch)')
plt.ylim(0, 1); plt.legend(); plt.show()

## Step 4, *Why* switching wins (the idea that matters)

When you first picked, your door had a **1/3** chance, and the other two doors held the remaining **2/3** between them. The host then opened one of those two, always a goat. He didn't change your door's 1/3, he **squeezed the whole 2/3 onto the single door he left closed.**

The number moved because **information arrived.** Before the reveal, P(car behind door 2) = 1/3. After it, P(car behind door 2 **given** the host opened door 3) = 2/3. Same door, new probability, because you *conditioned on what the host did.*

## Step 5, What this has to do with today

You just estimated a probability by **counting outcomes** over many games. A **classifier** does the same thing from data: it learns **P(yes | the features)**, the probability of an outcome *given what you know about a case*, from thousands of labelled examples.

And the deep link is that word **given**. Monty Hall turns a base rate (1/3) into an updated probability (2/3) by *conditioning on information*. Logistic regression is a machine that does exactly that, at scale: it turns a base rate (say, 30% of loans default) into a personalised probability (this applicant: 8%, that one: 61%) by conditioning on the features.

> **The habit to carry all day:** don't trust a felt probability. Compute it. In a minute we'll ask the room to guess the chance of an O-ring failing at 31°F, hold onto how wrong your gut was here.

## Stretch, the reveal only works because it's *informative*

Here's the subtle part. Switching wins 2/3 **because the host knows where the car is** and always opens a goat. What if the host opened one of the other doors **at random**, sometimes revealing the car? Change the simulation and see.

In [ ]:
def play_random_host(switch):
    doors = [0, 0, 0]
    doors[random.randint(0, 2)] = 1
    pick = random.randint(0, 2)
    host = random.choice([d for d in range(3) if d != pick])  # may reveal the car
    if doors[host] == 1:
        return None                      # host revealed the car, this round can't happen
    if switch:
        pick = [d for d in range(3) if d != pick and d != host][0]
    return doors[pick] == 1

res = [play_random_host(True) for _ in range(20_000)]
valid = [r for r in res if r is not None]
print(f'Random host, switching wins {sum(valid)/len(valid):.1%} of valid games  (now ~50/50)')

With a clueless host, switching is only 50/50. The reveal helped **only because of how it was generated**, the host's knowledge is baked into the data. Same lesson as the German credit case you'll meet later: *how the data came to exist* decides what you can honestly conclude from it.

**Do it conversationally?** *"Simulate Monty Hall 10,000 times for switch vs stay, plot how the win rate converges, then redo it with a host who opens a random door. Explain the difference."*